# Uso de script de modularización

### Paso 1 — Carga del panel de ventanas electorales y Revisión de variables



In [1]:
  %load_ext autoreload
  %autoreload 2

In [ ]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
from ml_models.cargar_panel import cargar_panel, columnas_candidatas 
from ml_models.lasso import *
NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES: 
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df)

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc)} variables _vc, N={len(df)}")
corr_por_nivel["municipal"] 

### Paso 2 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [4]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas _vc, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")



Nivel: municipal
De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> queda: emae_nivel_vc
cluster (8): ['ipc_final_vc', 'ipc_nivel_vc', 'ipc_pendiente_vc', 'ipc_volatilidad_vc', 'tc_oficial_final_vc', 'tc_oficial_nivel_vc', 'tc_oficial_pendiente_vc', 'tc_oficial_volatilidad_vc'] -> queda: ipc_nivel_vc
cluster (2): ['resultado_fiscal_final_vc', 'resultado_fiscal_nivel_vc'] -> queda: resultado_fiscal_nivel_vc
cluster (3): ['resultado_fiscal_volatilidad_vc', 'salario_real_final_vc', 'salario_real_nivel_vc'] -> queda: salario_real_nivel_vc


Nivel: provincial
De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> qu

In [5]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_v")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
municipal: N=11, P=24
[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
provincial: N=11, P=24
[nacional] excluye 1 fila(s) por NaN: ['nacional_2013_2015']
nacional: N=6, P=25


In [6]:
faltantes = columnas_nan("nacional", "nacional_2013_2015", columnas_finales_por_nivel["nacional"], paneles)
print("Columna(s) que rompen nacional_2013_2015:", faltantes)

Columna(s) que rompen nacional_2013_2015: ['resultado_fiscal_final_vc']


In [7]:
for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, columnas_finales_por_nivel[nivel],paneles)
    print(f"{id_t}: NaN en -> {faltantes}")


municipal_2001_2003: NaN en -> ['emae_nivel_vc', 'emae_pendiente_vc', 'emae_volatilidad_vc']
provincial_2001_2003: NaN en -> ['emae_nivel_vc', 'emae_pendiente_vc', 'emae_volatilidad_vc']


In [8]:
for nivel in NIVELES:
    columnas_finales_por_nivel[nivel] = [c for c in columnas_finales_por_nivel[nivel] if not c.startswith("emae_")]  
columnas_finales_por_nivel["nacional"] = [c for c in columnas_finales_por_nivel["nacional"] if not c.startswith("resultado_fiscal_final_vc")]
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_v")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

municipal: N=12, P=21
provincial: N=12, P=21
nacional: N=7, P=20


### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [9]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    # Chequeo 2: alpha=0 debe coincidir con OLS
    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

    residuo_manual = y_centrado - X_std @ beta_alpha_cero
    residuo_ols = y_centrado - X_std @ beta_ols

    print(f"{nivel} - Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
    print(f"{nivel} - Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

municipal - KKT: {'error_max_en_activos': np.float64(5.235086024679703e-07), 'exceso_max_en_inactivos': np.float64(-0.10473658184170365), 'n_activos': np.int64(8)}
municipal - Máxima diferencia vs. OLS (alpha=0): 3.391581537103192
municipal - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 4.891921480343342e-06
municipal - Residuo OLS (debería ser ~0 también): 3.9745984281580604e-14
provincial - KKT: {'error_max_en_activos': np.float64(6.117398873017876e-07), 'exceso_max_en_inactivos': np.float64(-0.08800592961126286), 'n_activos': np.int64(8)}
provincial - Máxima diferencia vs. OLS (alpha=0): 4.484152704128306
provincial - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 5.159574167024061e-06
provincial - Residuo OLS (debería ser ~0 también): 4.374278717023117e-14
nacional - KKT: {'error_max_en_activos': np.float64(4.17605542701871e-07), 'exceso_max_en_inactivos': np.float64(-0.20503214415191073), 'n_activos': np.int64(6)}
nacional - Máxima diferencia 

### Paso 4 — Grilla de alpha + LOO-CV manual

In [10]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=3.8109  alpha_1se=7.7118  (techo grilla=31.5791)


provincial: alpha_min=5.4508  alpha_1se=29.5909  (techo grilla=29.5909)


nacional: alpha_min=11.8511  alpha_1se=27.6125  (techo grilla=27.6125)


In [11]:

#validacion
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"{nivel}: X_df.shape={X_df.shape}, len(y_ser)={len(y_ser)}, 'resultado_fiscal_final_vc' in cols: {'resultado_fiscal_final_vc' in X_df.columns}")
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: X_df.shape=(12, 21), len(y_ser)=12, 'resultado_fiscal_final_vc' in cols: False


municipal: alpha_min=3.8109  alpha_1se=7.7118  (techo grilla=31.5791)
provincial: X_df.shape=(12, 21), len(y_ser)=12, 'resultado_fiscal_final_vc' in cols: False


provincial: alpha_min=5.4508  alpha_1se=29.5909  (techo grilla=29.5909)
nacional: X_df.shape=(7, 20), len(y_ser)=7, 'resultado_fiscal_final_vc' in cols: False


nacional: alpha_min=11.8511  alpha_1se=27.6125  (techo grilla=27.6125)


In [12]:
for nivel in ["provincial", "nacional"]:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- provincial ---


   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   9.863624   4.874320   9.863624  184.650964    215.379817
1                 3  29.590871   5.450845  29.590871  183.982974    214.044218
2                10  98.636236   5.108839  98.636236  182.239166    214.044218

--- nacional ---


   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   9.204160   9.204160   9.204160  188.372059    188.372059
1                 3  27.612480  11.851095  27.612480  175.338629    175.338629
2                10  92.041601  11.107513  92.041601  175.310080    175.338629



### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +40.7% | +13.3% |
| Provincial | +14.0% | +0.0% |
| Nacional | +0.0% | +0.0% |

Coeficientes en `alpha_min`: `icg_pendiente_vc` sobrevive en municipal (6.41) y provincial (4.41), `reservas_pendiente_vc` solo en municipal (0.96, débil). En `alpha_1se`: solo `icg_pendiente_vc` en municipal (2.81), nada en provincial ni nacional.

In [13]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal     204.688614           40.743299           13.331969
provincial    214.044218           14.044408            0.000000
nacional      175.338629            0.000000            0.000000


In [14]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
desocupacion_final_vc,NaN,NaN,0.0
desocupacion_pendiente_vc,NaN,NaN,0.0
desocupacion_volatilidad_vc,0.000000,0.000000,NaN
icg_pendiente_vc,6.409051,4.412778,0.0
reservas_final_vc,0.000000,0.000000,NaN
reservas_pendiente_vc,0.957210,0.000000,NaN


### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha_1se=7.71):** Signo estable (`icg_pendiente_vc` siempre positivo), magnitud sensible a `2009_2011` y `2011_2013` (caída a 0.48/0.78 vs. 2.3-3.7 en el resto). Composición casi estable (`reservas_pendiente_vc` se activa débilmente solo al sacar `2005_2007`).

**Provincial (alpha_min=5.45 -- alpha_1se no tuvo sobrevivientes):**
- Signo estable: `icg_pendiente_vc` positivo en las 12 corridas, nunca en cero.
- Magnitud más variable que en municipal: rango 1.41 (sacando `2009_2011`) a 5.55 (sacando `2005_2007`). **`2009_2011` vuelve a ser la ventana de mayor apalancamiento, igual que en municipal** -- coincidencia entre niveles que sugiere que esa transición puntual tiene un peso real en el vínculo confianza-voto.
- Composición menos estable que en municipal: sacar `2015_2017` activa `icc_pendiente_vc` (2.02) mientras `icg` cae a 2.18 -- acá sí cambia cuál variable "aporta", no solo cuánto. `reservas_pendiente_vc`/`reservas_volatilidad_vc` aparecen de forma esporádica y débil en varias otras corridas, sin patrón consistente -- lectura: ruido, no señal.

**Nacional (alpha_min=11.85):** ninguna variable sobrevive en ninguna de las 7 corridas -- resultado nulo robusto.

**Síntesis:** `icg_pendiente_vc` es el hallazgo más sólido del ejercicio de LASSO -- signo positivo y consistente en municipal y provincial, con la particularidad de que la transición `2009_2011` reduce su magnitud en ambos niveles simultáneamente. Provincial es menos estable en composición que municipal (coherente con su menor mejora sobre baseline, 14% vs. 40.7%). Nacional no tiene ninguna señal individual bajo ningún criterio.

In [15]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_min", resultados_cv["provincial"]["alpha_min"]),
    "nacional": ("alpha_min", resultados_cv["nacional"]["alpha_min"]),
}

for nivel in NIVELES:
    df = paneles[nivel]
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    X_df, y_ser = datos_final[nivel] 
    resultado = estabilidad_seleccion(nivel, alpha, df, columnas_finales_por_nivel[nivel], "delta_v", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha=7.712, criterio=alpha_1se) ---
                         icg_pendiente_vc  reservas_pendiente_vc
sin_municipal_2001_2003          3.280685               0.000000
sin_municipal_2003_2005          3.280341               0.000000
sin_municipal_2005_2007          2.937138               0.150497
sin_municipal_2007_2009          3.657110               0.000000
sin_municipal_2009_2011          0.478146               0.000000
sin_municipal_2011_2013          0.784155               0.000000
sin_municipal_2013_2015          2.820476               0.000000
sin_municipal_2015_2017          2.321007               0.000000
sin_municipal_2017_2019          3.589671               0.000000
sin_municipal_2019_2021          3.470692               0.000000
sin_municipal_2021_2023          3.112734               0.000000
sin_municipal_2023_2025          3.203493               0.000000

--- provincial (alpha=5.451, criterio=alpha_min) ---
                          icc_pendiente_vc  icg_p

In [16]:
import importlib
import ml_models.lasso
importlib.reload(ml_models.lasso)
from ml_models.lasso import construir_Xy_final
